[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2_orientation.ipynb)
# Rationale 2 Orientation: Detecting Plant Health

##### Welcome. This notebook orients you to PlantVillage, the image dataset that anchors all four Rationale 2 questions.

#### By the end you will have:

- **A Colab environment set up with the `irilab2026` library and a sample of the PlantVillage dataset cached locally.**
- **A sense of what PlantVillage is, how it was built, and why one specific fact about it (Noyan 2022) shapes most of the Rationale 2 work you'll do.**
- **An overview of lab-condition disease photographs — six to eight leaves spanning the dataset.**
- **A sense of the dataset's class structure: 38 host-disease classes, ~54,000 images total, and the class imbalance you'll need to consider.**
- **A clear handoff to Rationale 2 Question 1, where PlantDoc enters and the comparison with PlantVillage begins.**

You don't need to install anything on your computer. Everything runs in Colab. To begin, click the cells in order from top to bottom and run each one. We'll explain what each cell does as we go.

#### What It Is and Who Built It
**PlantVillage** is a research dataset containing approximately 54,000 leaf images across 38 classes, representing 14 host plant species and their diseases. It was introduced in the Mohanty, Hughes & Salathé 2016 paper that proposed using deep learning for smartphone-based plant disease diagnosis in low-resource agricultural settings. The project's main goal was to create "a tool a farmer could use in the field." 

In the lab, plant leaves were removed, placed against solid backgrounds (usually gray or black), and photographed under controlled lighting, with each image capturing a single leaf. This approach was intentional: clear, uniform images provide better training data, illustrating concretely what "lab-photographed" means. The design choices of **PlantVillage** have implications that the original paper does not address. Noyan 2022 demonstrated that a classifier trained on just eight background pixels—without any leaf—predicts PlantVillage class labels with about 49% accuracy. This is significant, considering chance performance is below 3% across 38 roughly balanced classes. This isn’t a flaw but a characteristic of the dataset—its lab-condition design and the background's predictability reflect the same underlying aspect from different perspectives. 

The questions in Rationale 2 are centered on this fact. R2-Q1 explores whether a model trained on lab-condition PlantVillage images can transfer to field photographs of the same diseases (PlantDoc). R2-Q2 investigates what the model attends to when it makes errors. R2-Q3 considers whether targeted augmentation can reduce the gap. R2-Q4 examines whether the model learns the disease or the host plant. Collectively, these questions form a coordinated effort to understand what a classifier trained on this specific dataset has actually learned. The section concludes by preparing you to analyze the dataset directly in the rest of this notebook.

## Setup
Before we look at any data, we need to install the lab's shared library, set up the working environment, and confirm that everything is in place. The `irilab2026` library bundles four small helpers that handle the chores you'd otherwise do by hand at the start of every notebook, plus a wrapper function that runs them in one shot. We'll meet the four helpers individually first, then use the wrapper.

In [ ]:
!pip install git+https://github.com/geraldmc/irilab2026.git -q

In [ ]:
import irilab2026 as iri

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

### The four environment helpers

The library exposes four functions that each handle one piece of getting a Colab notebook ready to do real work:

- `is_colab()` returns `True` if you're running in Google Colab, `False` otherwise. Useful for code that should behave differently depending on where it runs.
- `has_gpu()` returns `True` if a GPU is attached to the runtime. Image-model training is impractical without one; orientation work like this notebook doesn't need one.
- `cache_dir()` returns the directory where downloaded data will be stored. On Colab with Google Drive mounted, this is a folder on your Drive — so cached data survives a runtime reset. Locally, it's a folder in your home directory.
- `mount_google_drive()` mounts your Drive at `/content/drive` when you're on Colab, so that downloads persist. Has no effect locally. We won't call this directly here; the wrapper function in a moment will handle it for us.

Let's run the first three and see what they return on this runtime.

In [ ]:
print(f"Running in Colab: {iri.is_colab()}")
print(f"GPU attached:     {iri.has_gpu()}")
print(f"Cache directory:  {iri.cache_dir()}")

### The `setup()` wrapper

Calling those helpers by hand at the top of every notebook is repetitive. The library wraps them in a single function — `setup()` — that runs all four (including the Drive mount on Colab) and prints a status summary so you can see what it found.

`setup()` takes one parameter worth knowing about: `gpu_required`. When set to `True`, the function raises an error if no GPU is detected, stopping the notebook before you waste time on a session that can't run. When `False`, it prints a note and continues. This orientation notebook does no model training, so we set `gpu_required=False` — later notebooks (R2-Q1 onward) will set it to `True`.

In [ ]:
iri.setup(gpu_required=False)

The environment is ready. The next section fetches the orientation slice of PlantVillage and shows you what's inside.

### Loading the data

The loader function fetches a curated slice of PlantVillage from a versioned release on GitHub, verifies the file's checksum against a known hash, extracts it into the cache directory we set up in the previous section, and returns a dictionary with three things to work with.

The download happens once. On first run, you'll see progress messages — fetching, verifying, extracting. On subsequent runs in the same session (or in any later session, as long as the cache survives), the function notices the data is already there and returns immediately.

In [ ]:
data = iri.load_plantvillage_orientation()

What did we get back? A dictionary. Let's see its keys.

In [ ]:
data.keys()

### What's in the dictionary

`data['manifest']` is a 38-row pandas DataFrame describing the **full** PlantVillage dataset. One row per class. The full dataset has roughly 54,000 images across those 38 classes; this DataFrame tells us about every one of them, even though our download is a small sample. Section 6 uses the manifest to map the dataset's structure.

`data['sample_paths']` is a 190-row pandas DataFrame, where each row points to one image file actually on disk. Five images per class, chosen deterministically — the same images every time the tarball is built. Section 5 uses `sample_paths` to open and display actual leaves.

`data['sample_dir']` is the directory those 190 image files live in. Most of the time you'll work through `sample_paths` rather than the directory directly, but it's there if you need it.

Let's look at the manifest's structure.

In [ ]:
data['manifest'].head()

Five columns: `class` (the class label), `host` (the plant species), `disease` (the disease name, or `healthy` for the no-disease classes), `is_healthy` (a boolean flag for convenience), and `n_total` (the count of images in that class in the full dataset).

The asymmetry between the manifest and `sample_paths` is worth flagging before we move on. The manifest has 38 rows describing roughly 54,000 images; `sample_paths` has 190 rows describing 190 image files actually on your disk. Both are right — they describe different things. Total class counts and per-class image counts come from the manifest, because they're facts about the full dataset. Visual inspection comes from `sample_paths`, because it points to bytes you can actually open.

Section 5 uses `sample_paths`. Section 6 uses the manifest. One ~3 MB tarball, two complementary views.

## Looking at the leaves

We have 190 image files on disk — five from each of the 38 classes. Let's pick eight of them, one from each of the first eight classes, and put them on screen.

In [ ]:
to_display = data['sample_paths'].groupby('class').head(1).head(8)
to_display

Eight rows, eight images. We'll open each image and lay them out in a 2×4 grid. The class names follow PlantVillage's `Host___Disease` format with triple underscores between host and disease — you'll see those in the titles.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
paths = to_display['path'].tolist()
classes = to_display['class'].tolist()

for ax, path, cls in zip(axes.flat, paths, classes):
    ax.imshow(Image.open(path))
    ax.set_title(cls, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Now look at the backgrounds

You probably looked at the leaves first. That's where the eye goes — they're the foreground, they're what each label names. Look again, but this time at the backgrounds.

The backgrounds are mostly solid — gray, dark gray, near-black. At a glance they look uniform. But class to class, they're not quite the same. Different shades. Different lighting. Some have a faint tabletop edge. Some are slightly warmer or cooler. The variation is subtle, but it's there — and it has nothing to do with the disease the label names.

This is what Section 2 was pointing at. Noyan (2022) showed that a classifier could predict PlantVillage class labels from just eight background pixels at about 49% accuracy. You've now seen what that classifier was reading: not the leaves, but the systematic background variation between classes. The signal isn't hidden — it just isn't where you'd look first.

The cleanness of the photography is what makes PlantVillage so trainable. Solid backgrounds, controlled lighting, single-leaf framing strip out every confound a real-world image would carry. That same cleanness is what makes the backgrounds informative about the labels. The dataset is clean because of how it was made, and informative in the wrong places for the same reason. One fact, two faces.

## What's actually in here

Section 5 looked at eight images. The dataset has roughly 54,000 of them across 38 classes — to reason about it as a whole, we need the manifest, not the disk. This section uses the manifest to build a structural map: how many classes there are, how they break down by host plant, how many images per class, and the imbalance pattern that downstream R2 work has to grapple with.

### Size

Two basic facts to start: how many classes the dataset defines, and how many images it has across all of them.

In [ ]:
print(f"Number of classes: {len(data['manifest'])}")
print(f"Total images across all classes: {data['manifest']['n_total'].sum():,}")

### Healthy versus diseased

PlantVillage includes both diseased-leaf classes (a leaf showing one specific disease on one specific host) and healthy-leaf classes (a leaf of one host with no disease). The manifest carries an `is_healthy` boolean flag to distinguish them.

In [ ]:
manifest = data['manifest']
healthy_classes = manifest['is_healthy'].sum()
diseased_classes = len(manifest) - healthy_classes

print(f"Healthy-leaf classes: {healthy_classes}")
print(f"Diseased-leaf classes: {diseased_classes}")

### Host coverage

The 38 classes span 14 host plant species. The classes-per-host count varies — some hosts have one healthy class plus several disease classes; others have only one or two classes total. Tomato is the most heavily represented; several hosts have just a handful.

In [ ]:
manifest.groupby('host').size().sort_values(ascending=False)

### Image counts per class

Class counts and host coverage tell us *what* is in the dataset. The per-class image-count distribution tells us how *evenly* it's distributed — and that's where the dataset gets less uniform than it sounds.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plot_data = (
    manifest[['class', 'n_total']]
    .sort_values('n_total', ascending=False)
    .reset_index(drop=True)
)

plot_data['rank'] = plot_data.index + 1

# Assign threshold categories
conditions = [
    plot_data['n_total'] > 3000,
    plot_data['n_total'] > 1500,
    plot_data['n_total'] > 1000
]

choices = [
    '> 3000 images',
    '1501–3000 images',
    '1001–1500 images'
]

plot_data['count_group'] = np.select(
    conditions,
    choices,
    default='≤ 1000 images'
)

# Muted pastel palette
palette = {
    '> 3000 images': '#A8DADC',      # muted pastel blue
    '1501–3000 images': '#B8E0D2',  # muted pastel green
    '1001–1500 images': '#F6D6AD',  # muted pastel peach
    '≤ 1000 images': '#D8C7E8'      # muted pastel lavender
}

fig, ax = plt.subplots(figsize=(12, 5))

sns.barplot(
    data=plot_data,
    x='rank',
    y='n_total',
    hue='count_group',
    palette=palette,
    dodge=False,
    ax=ax
)

ax.set_xlabel('Class rank, sorted by number of images')
ax.set_ylabel('Number of images')
ax.set_title('PlantVillage: class imbalance by sorted class frequency')

# Show fewer x-axis labels
step = max(1, len(plot_data) // 10)
ax.set_xticks(plot_data.index[::step])
ax.set_xticklabels(plot_data['rank'].iloc[::step])

ax.legend(title='Class size group', loc='upper right')

plt.tight_layout()
plt.show()

### What the imbalance means for downstream work

The largest class is several times the size of the smallest. That isn't pathological — it reflects how much imagery the original collection effort gathered for each host-disease pairing — but it's structural information that the rest of Rationale 2 has to take into account.

The clearest example is R2-Q1, which compares a model's accuracy on PlantVillage's own held-out images against its accuracy on PlantDoc, the field-photograph counterpart. PlantDoc is much smaller than PV (roughly 2,600 images across 27 classes, leaving 8–12 test images per class), and its class space doesn't line up perfectly with PV's. Per-class accuracy on counts that small is too noisy to support reliable claims, so R2-Q1 aggregates — by host group, or by disease type (fungal, bacterial, viral) — and reports group-level accuracy rather than per-class. The imbalance you just saw on the PV side is one half of why that aggregation matters; the small per-class test counts on the PD side are the other half.

The point of this section isn't to teach R2-Q1. It's to plant the instinct that imbalance and class structure are not background facts about a dataset — they're inputs to every claim a model makes when trained on it. Section 7 hands off to where that work begins.

## Where to next

You started this notebook with PlantVillage as a dataset name. You finish it with three things: a vivid sense of what its lab-condition photography actually looks like, a structural map of its 38 classes and their imbalanced distribution, and — if Section 5's exercise landed — the beginning of an instinct to look at what's *around* a labeled subject, not just at it.

That instinct is the character of Rationale 2. The four questions you'll work on don't take a model's accuracy at face value; they ask what produced it. R2-Q1 measures how a model trained on PlantVillage's clean indoor images performs on PlantDoc — a much smaller dataset of leaves photographed in real fields, on living plants, with cluttered backgrounds and natural lighting. R2-Q2 looks at the cases where the PV-trained model gets things wrong on PlantDoc and asks what the model was attending to when it failed. R2-Q3 tests whether targeted training adjustments can close the gap R2-Q1 measures. R2-Q4 takes a different cut: when the model gets things *right* on PlantVillage itself, has it learned the disease, or has it learned the host plant?

The four questions are independent investigations, but they share that diagnostic posture — accuracy is a number, and the work is figuring out what that number is made of.

### When you're ready

Open the R2-Q1 question page on Notion to see the full framing of the first question, the prediction it asks you to make in advance, and the week-by-week workflow. The R2-Q1 analytical notebook (when it's ready) will pick up where this one leaves off — same library, same cache, expanded loader for the full PlantVillage dataset and the introduction of PlantDoc.

If you want to revisit anything from this notebook, the cache is persistent: re-running this notebook in a new session will skip the download and pick up from the same data you've already verified.